# Imports

In [ ]:
!pip install osmnx

In [ ]:
from collections import defaultdict
import math
from math import radians, sin, cos, sqrt, atan2

from geopy.distance import geodesic

import geopandas as gpd
import osmnx as ox
from pyproj import Transformer
import pyproj
import folium
import os

from shapely.geometry import Polygon as ShapelyPolygon, Point, LineString
from ipyleaflet import Map, Polygon, Marker, Polyline, MeasureControl
from ipywidgets import Layout


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_FOLDER = "/content/drive/MyDrive/ng_demand_points"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Classes

In [ ]:
class Points:
  def __init__(self):
    self._demand_points = []  # array of dicts: [{'demand_zone': demand_zone, 'x':x, 'y':y, 'q':q}, {'demand_zone': demand_zone, 'x':x, 'y':y, 'q':q}, ...]
    self._demand_zone_centers = defaultdict(float) #[{'demand_zone': demand_zone, 'x':x, 'y':y}, {'demand_zone': demand_zone, 'x':x, 'y':y}, ...]
    self._q_in_demand_zone = defaultdict(float) #[{'demand_zone': demand_zone, 'q':q}, {'demand_zone': demand_zone,'q':q}, ...]
    self._area_in_demand_zone = defaultdict(float) #[{'demand_zone': demand_zone, 'area':area}, {'demand_zone': demand_zone,'area':area}, ...]

    self.__centers_dirty = False #checks if zone demand center needs to be calcualted again.

  def add_demand_point(self, demand_zone, x, y, q) -> None: #adds to 'demand_points'.
    #checking for
    if None in (demand_zone, x, y, q):
      raise ValueError(f"Missing values. Received: zone={demand_zone}, x={x}, y={y}, q={q}")

    demand_point = {'demand_zone': demand_zone, 'x':x, 'y':y, 'q':q}
    self._demand_points.append(demand_point)

    self.__centers_dirty = True


  def add_area_in_demand_zone(self, zone: int, area: int) -> None:
      self._area_in_demand_zone[zone] = area

  #properties for accessing class variables safely.
  @property
  def demand_zone_centers(self):
      if self.__centers_dirty:
          self.__calculate_zone_demand_centers()
          self.__centers_dirty = False

      return self._demand_zone_centers

  @property
  def q_in_demand_zone(self):
    return self._q_in_demand_zone

  @property
  def area_in_demand_zone(self):
    return self._area_in_demand_zone




  #helper function used to calculate zone demand centers.
  def __calculate_zone_demand_centers(self): #calculates and adds to 'demand_zone_centers'.
    q_x_nominator = defaultdict(float)
    q_y_nominator = defaultdict(float)

    self._q_in_demand_zone.clear()
    self._demand_zone_centers.clear()

    for demand_point in self._demand_points:
      zone = demand_point['demand_zone']
      q_x_nominator[zone] += demand_point['x'] * demand_point['q']
      q_y_nominator[zone] += demand_point['y'] * demand_point['q']
      self._q_in_demand_zone[zone] += demand_point['q']

    for zone, total_q in self._q_in_demand_zone.items():
      if total_q == 0:
          continue # Prevent ZeroDivisionError if a zone has exactly 0 population.

      center_x = q_x_nominator[zone] / total_q
      center_y = q_y_nominator[zone] / total_q

      self._demand_zone_centers[zone] = (center_x, center_y)



In [ ]:
class Network:
  def __init__(self):
    self._gravitating = 0.8
    self._attracting = 0.4
    self._L = 0
    self._K = 0
    self._network_inertia = 0
    self._all_of_demand_zones_q = 0
    self._gravitating_demand_zones_area = 0

    self._network_nodes = []

    self.__network_dirty = False

    self.transformer = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)

  def add_node(self, node: tuple):
    self._network_nodes.append(node)
    self.__network_dirty = True

    # Only create the route if we have at least 2 points
    if len(self._network_nodes) >= 2:
        # Project ALL current nodes to meters
        projected_nodes = [self.transformer.transform(lon, lat) for lat, lon in self._network_nodes]
        # Update the geometry object
        self.geo_route = LineString(projected_nodes)
        # Update the length in km.
        self._L = (self.geo_route.length/1000)

  def calculate_network(self, points: Points) -> None:
    if self.__network_dirty:
        self.__calculate_network(points)
        self.__network_dirty = False

  @property
  def gravitating(self):
    return self._gravitating
  @property
  def attracting(self):
    return self._attracting
  @property
  def L(self):
    return self._L
  @property
  def K(self):
    return self._K
  @property
  def network_inertia(self):
    return self._network_inertia
  @property
  def all_of_demand_zones_q(self):
    return self._all_of_demand_zones_q
  @property
  def gravitating_demand_zones_area(self):
    return self._gravitating_demand_zones_area


  def __calculate_network(self, points: Points) -> None:
        """
        Calculates inertia by finding the distance from each neighborhood
        center to the nearest point on the transit route.
        """
        self._K = 0
        self._network_inertia = 0
        self._all_of_demand_zones_q = 0
        self._gravitating_demand_zones_area = 0

        for zone, center in points.demand_zone_centers.items():
            # center is (lat, lon)
            lat, lon = center
            q = points.q_in_demand_zone[zone]
            area = points.area_in_demand_zone[zone]
            self._all_of_demand_zones_q += q


            # Project the neighborhood center to meters
            point_m = Point(self.transformer.transform(lon, lat))

            # Find the shortest distance to the network in km.
            min_dist = (self.geo_route.distance(point_m) / 1000)

            # If within service distance D ( 1km ), add to inertia.
            if min_dist <= self._gravitating:
                contribution = q * (min_dist**2)
                self._gravitating_demand_zones_area+=area

                #print(f"Zone {zone}: Q = ({q}) * Dist({min_dist:.2f}km) = {contribution:.2f}")

                self._network_inertia += contribution
                self._K += 1

In [ ]:
# @title neighborhoods_data, hood_population, kav_3_nodes
hood_border = [
    [(31.234489775441425, 34.794527700154624), (31.234122826079773, 34.78916178312236), (31.23217797066964, 34.78529832285912), (31.23188440411147, 34.781949990631006), (31.233407688377273, 34.780527178591896), (31.23344438365532, 34.77855252112404), (31.232838909745066, 34.77707152802312), (31.230710546421662, 34.77644908164736), (31.23023349289292, 34.77743641038133), (31.22674684145421, 34.77821287799653), (31.22843492848417, 34.781089009525836), (31.227223912584574, 34.78769981930958), (31.227260610264093, 34.79619943188867), (31.229132173023007, 34.798431653374074), (31.230233074978393, 34.80276731433613), (31.2327651008011, 34.80182291293845), (31.232618318548134, 34.79856043538285), (31.233792987980685, 34.79645128962845), (31.2344718455491, 34.79486297818688)],
    [(31.241296845480033, 34.76799724118051), (31.240948268739135, 34.762137659781295), (31.237590858477894, 34.76203034144064), (31.234196633916092, 34.76084983969356), (31.233077430435095, 34.7622020507857), (31.234031506367412, 34.766795275765304), (31.235957976477927, 34.77035824467474), (31.238563235474004, 34.768340659870596), (31.2410766918986, 34.76804016851679)],
    [(31.23423291132462, 34.77400361263131), (31.235847475973856, 34.770483571058165), (31.233865960965343, 34.76662011079494), (31.233132055971232, 34.76327177856682), (31.233462313924047, 34.760438574373765), (31.232691710238417, 34.76009515568372), (31.2263064665009, 34.761941031142804), (31.22315038211729, 34.76310006922176), (31.222342994765334, 34.76563278206101), (31.2257559942545, 34.76863769559905), (31.22740740137495, 34.770741135075724), (31.22905877963859, 34.773016283897384), (31.23104039546277, 34.77374604861376), (31.233645790065687, 34.77606412477171), (31.23401274128005, 34.77438995865764)],
    [(31.242415533771293, 34.77624065490869), (31.239810381022235, 34.778429949057866), (31.238141261461358, 34.77803937177219), (31.236343265466914, 34.777159361378914), (31.234581930088243, 34.776880333693235), (31.234398455764556, 34.773961274827684), (31.2358662403776, 34.77039830591826), (31.238434808578123, 34.76829486644162), (31.241296845160907, 34.767951447751535), (31.24149865215977, 34.77123538897529), (31.24236091357063, 34.77604325063621)],
    [(31.262043598420373, 34.76378550353558), (31.26461145460694, 34.766146507029774), (31.2664455948178, 34.76855043786025), (31.268829923793547, 34.7671338357637), (31.272167883134422, 34.76352793951803), (31.275138934408595, 34.75983618859986), (31.278256603641225, 34.75266732344476), (31.2727180848746, 34.749104354535326), (31.27029717321529, 34.74738726108502), (31.26582199114608, 34.74974826457921), (31.264831553314565, 34.752796105453534), (31.26362100406982, 34.75459905357637), (31.26020937265203, 34.75502832693895), (31.256907676439603, 34.758848859865935), (31.261566703149903, 34.7636567215268)],
    [(31.261016436052735, 34.752926578337075), (31.259182190321987, 34.7564895472465), (31.256870989948233, 34.75889347807698), (31.25393605077583, 34.75636076523772), (31.255807085041695, 34.753613415717204), (31.26094306690793, 34.752282668293226)],
    [(31.252028292887758, 34.770183247592634), (31.254119488652144, 34.761769489686046), (31.256687560427977, 34.758850430820495), (31.25455973448959, 34.75674699134386), (31.251257840666707, 34.755330389247355), (31.24791914172565, 34.756103081299976), (31.24546090331384, 34.757476756060235), (31.24248891770041, 34.75902214016553), (31.241094620615748, 34.758979212829274), (31.24098454365347, 34.76219876304862), (31.241314774155377, 34.76803688077971), (31.247845762401163, 34.768552008814815), (31.25177147617931, 34.770054465583854)],
    [(31.26651895970931, 34.768640117987054), (31.263841105112153, 34.76542056776771), (31.25936561685293, 34.76129954348693), (31.256981048745903, 34.758938539992734), (31.25437629782255, 34.76087027012436), (31.253018863925533, 34.76584984113029), (31.251991603678338, 34.770400138773645), (31.25617395051085, 34.77383432567426), (31.258375111261426, 34.77598069248718), (31.26336421892494, 34.770614775454916), (31.266188817359073, 34.768768899995834)],
    [(31.251954916405058, 34.78679950847406), (31.25257861170194, 34.784738996333665), (31.254486359974383, 34.782850193538344), (31.25837511206079, 34.77585303772826), (31.255917145879195, 34.773363252225295), (31.251954916405058, 34.77035833868722), (31.249496783058422, 34.775294982356904), (31.24703858571661, 34.778643314585025), (31.246304783063028, 34.782936048210836), (31.250267249649855, 34.78448143231612), (31.251661411310266, 34.78667072646528)],
    [(31.246341473125117, 34.782972463880355), (31.244286796158626, 34.782156844491446), (31.242121998852923, 34.77563188938023), (31.241278081376763, 34.76794789619005), (31.249203270116258, 34.7688064429152), (31.25199160427175, 34.770308899684245), (31.250524070243898, 34.774000650602446), (31.24854286312402, 34.77657629077791), (31.247075275493593, 34.77850802090953), (31.24641485361857, 34.78237148117275)],
    [(31.27165435824347, 34.78143525833254), (31.27099410832552, 34.77722837937927), (31.26916005653123, 34.772935645753456), (31.26648227688672, 34.768514130118874), (31.261640071438222, 34.77194831701953), (31.257311222911305, 34.777013742697996), (31.254449671675413, 34.78302356977411), (31.252285107350303, 34.78525579125952), (31.25191822712233, 34.786972884709826), (31.25789819705831, 34.78770264942625), (31.260906382271614, 34.78808899545253), (31.263584320145284, 34.78950559754909), (31.266115451839188, 34.788046068116294), (31.267656107455785, 34.78589970130341), (31.26905001228979, 34.783452843136686), (31.271360914405957, 34.781478185668824)],
    [(31.258375111200973, 34.79837909644595), (31.25830174000293, 34.7917682866622), (31.258925393368422, 34.7876901897177), (31.26097975178078, 34.7882911724253), (31.262593859163733, 34.78914971915045), (31.264611454571114, 34.789235573822985), (31.266298864877047, 34.787647262381455), (31.26794956315216, 34.785329186223514), (31.269746957320297, 34.782367200021675), (31.271654358579287, 34.78129401661526), (31.27202116209258, 34.78455649417085), (31.272644724792166, 34.78781897172648), (31.273818478692228, 34.791939996007265), (31.274295312042437, 34.79468734552774), (31.27499221798772, 34.79618980229678), (31.273488361883985, 34.79756347705705), (31.27172771939605, 34.79790689574713), (31.271599754841322, 34.79968610963771), (31.268188411887735, 34.80161783976932), (31.26545558043508, 34.80333493321967), (31.26417167256067, 34.804408116626085), (31.26393323060354, 34.804257870949186), (31.26393323060354, 34.7982695075412), (31.25929266348647, 34.79826890794023), (31.25849496498748, 34.798215225747654)],
    [(31.26090638343366, 34.79821945107279), (31.258265055208753, 34.798305305745316), (31.255293472604528, 34.7987345791079), (31.253422428158952, 34.79912092513423), (31.25074420197464, 34.798305305745316), (31.250157183345262, 34.80053752723073), (31.249350026770795, 34.801009727929554), (31.247699003195923, 34.80298438539746), (31.246781755402893, 34.80380000478637), (31.244947233088308, 34.805001970201566), (31.24637816355142, 34.80796395640339), (31.247588933931176, 34.809509340508676), (31.25074420197464, 34.810968869941426), (31.251991604444164, 34.81178448933034), (31.253349053111346, 34.81444598417833), (31.25712779390393, 34.812385472037974), (31.259108820870857, 34.810668378587636), (31.260466167195982, 34.80869372111977), (31.260722960250824, 34.806375644961825), (31.26075964491592, 34.80444391483021), (31.260998511807777, 34.79855438733252)],
    [(31.25843055746044, 34.79827729036203), (31.255422293341137, 34.798749491060846), (31.254064874484378, 34.799028518746525), (31.251955333472214, 34.798792418397134), (31.25076296335401, 34.7984060723708), (31.2515334196134, 34.794134802413126), (31.251643484280084, 34.790786470185), (31.251753548818424, 34.78864010337212), (31.2519369894306, 34.78681569158112), (31.253734688567974, 34.78737374695247), (31.25769684334447, 34.78767423830631), (31.25905420997479, 34.78767423830631), (31.258944153948146, 34.78909084040282), (31.258393871890082, 34.79160208957391), (31.258412214677055, 34.795014812806436), (31.258412214677055, 34.798191435689496)],
    [(31.250781307600246, 34.79832094921149), (31.251643484256572, 34.793942360913185), (31.251643484256572, 34.79125940239703), (31.252028709579506, 34.78688081409874), (31.251276601535327, 34.78623690405485), (31.25074461905724, 34.78488469296271), (31.250047534031616, 34.78428371025512), (31.24878176122325, 34.783790045888125), (31.24632354526738, 34.782888571826724), (31.24555304649738, 34.78645154073616), (31.244984340991877, 34.789220353924804), (31.24436059553182, 34.79216087645847), (31.24382857408988, 34.793470160214326), (31.244819232300422, 34.79550920868659), (31.24628685499232, 34.79699020178751), (31.247681075423476, 34.79780582117641), (31.249222031950726, 34.798106312530216), (31.25074461905724, 34.798385340215894)],
    [(31.22120530096047, 34.77052887281432), (31.218342655780557, 34.774091841723745), (31.222563192078226, 34.77868506670335), (31.223113682920214, 34.782076326267756), (31.22682023778319, 34.781003142861294), (31.22612297618676, 34.777053827925535), (31.22505872488474, 34.77443526041383), (31.22177456740773, 34.77093338662701)],
    [(31.26101560164251, 34.81422669074612), (31.26424378757592, 34.81688818559411), (31.26618881664037, 34.814990992232175), (31.267876198739362, 34.817609559743914), (31.27077402367032, 34.81533441092226), (31.27246132376782, 34.81619295764741), (31.272791444170167, 34.819970563238115), (31.271580997047256, 34.827010646384444), (31.27770643351197, 34.825508189615405), (31.279503641785716, 34.821215455989595), (31.280622711916156, 34.8212972652791), (31.28111784924156, 34.81904358012556), (31.278018613564836, 34.81734795034334), (31.27730339086845, 34.81775576003779), (31.275744553379575, 34.81526597453482), (31.277596399405457, 34.81086627440932), (31.2774496868429, 34.8072174508274), (31.27642269251501, 34.80640183143849), (31.277192939309273, 34.80442717397063), (31.277413008666592, 34.80215202514893), (31.276642763669493, 34.79940467562845), (31.275799154627723, 34.797610997336875), (31.26838974382798, 34.80190373096268), (31.26905001197186, 34.80456522581067), (31.26743601501848, 34.808686250091455), (31.264354671442934, 34.80714086598618), (31.26123654278447, 34.81362289376115)],
    [(31.239186601392387, 34.797747250373966),(31.241424850347602, 34.796459430286234),(31.243883193816895, 34.79315402539436),(31.244873851454486, 34.78924763779489),(31.246268092747076, 34.78280853735616),(31.244506942429975, 34.78216462731228),(31.243369515395226, 34.77915971377424),(31.242415533771293, 34.776154800236164),(31.239847073813344, 34.77830116704909),(31.238232577520357, 34.77808653036778),(31.236397889147284, 34.77727091097887),(31.234563165142713, 34.77692749228882),(31.234636554787006, 34.780104115171916),(31.233462313637364, 34.782851464692406),(31.233242141797078, 34.78486904949655),(31.234599859971997, 34.78714419831821),(31.235517226071842, 34.789633983821176),(31.237278544013257, 34.793025243385586),(31.23874628387951, 34.79688870364881)],
    [(31.2841248777057, 34.81075009601334), (31.286912180063734, 34.808861293217966), (31.288855907941155, 34.80757347313024), (31.289185970958993, 34.805341251644826), (31.288269126389558, 34.8004046797514), (31.28808575640572, 34.79705627574701), (31.282401110014334, 34.79684163906575), (31.27913686784519, 34.79684163906575), (31.276092585131714, 34.79821531382596), (31.27708290474921, 34.800662171992684), (31.277523043463898, 34.80272268413308), (31.27675279936537, 34.80572759767116), (31.28060395695619, 34.808904220554254), (31.2839048240103, 34.81062131400456)]
]

hood_population = [5649, 9572, 9432, 30485, 14008, 3985, 4733, 16168, 12900, 10774, 22342, 15439, 22342, 9693, 7754, 2205, 13462, 7679, 18377]

kav_3_nodes = [(31.242893360407432, 34.79779958724976),(31.241370646801023, 34.797756671905525),(31.240434991324268, 34.79490280151368),(31.24014145044116, 34.794495105743415),(31.238306799254318, 34.79267120361329),(31.23997633329353, 34.79020357131959),(31.2420311039101, 34.792242050170906),(31.243856509953265, 34.79388356208802),(31.244654540707472, 34.79540705680848),(31.24614968362316, 34.796973466873176),(31.247011902518373, 34.79590058326722),(31.247635630428764, 34.794516563415534),(31.248369422695898, 34.79329347610474),(31.248947280092825, 34.79237079620362),(31.248305216100146, 34.792016744613655),(31.247626457989334, 34.790525436401374),(31.248598731609935, 34.78966712951661),(31.25179982590087, 34.79141592979432),(31.25206581452559, 34.78682398796082),(31.251304534740786, 34.78607296943665),(31.25076338031783, 34.78488206863404),(31.250341461158307, 34.7844421863556),(31.251286190573836, 34.78255391120911),(31.252570273652125, 34.78257536888123),(31.25338657452883, 34.78211402893067),(31.254496365523345, 34.780794382095344),(31.25620229979263, 34.77814435958863),(31.25711945601182, 34.776502847671516),(31.257018569263927, 34.775998592376716),(31.256358216978576, 34.77523684501649),(31.25836677416916, 34.772554636001594),(31.25968744601735, 34.770998954772956),(31.26103561280168, 34.77296233177186),(31.26240210233882, 34.77714657783509),(31.263731888596965, 34.78306889533997),(31.264126235402234, 34.78537559509278),(31.264062039522965, 34.78742480278016),(31.265015802376517, 34.786856174469),(31.26655647585351, 34.78462457656861),(31.267115880831394, 34.78370189666749),(31.267959567312037, 34.78540778160096),(31.268748223939344, 34.78416323661805),(31.269967877603264, 34.78244662284852),(31.27129755724196, 34.78155612945557),(31.27218705660747, 34.7814702987671)]

area_for_each_neighberhood_beer_sheva = [(1.4178724778055327),(0.5510378401699411),(1.2217553243177663),(0.6373353975978205),(2.5113366557582912),
(0.2548843434929496),(1.634614546000421),(1.4075602770552245),(1.0854111197191363),(0.984501296599488),(2.4397032023391105),(2.1739110747679864),(1.828598080301576),(0.845585876162048),(0.9509992193158466),
(0.49860962435037304),(2.717549603342069),(1.833653369632883),(1.4559562127577135)]


In [ ]:
def get_demand_points_from_neighberhood(hood_border, hood_population, hood_id):
    # ── All reads and writes go through Google Drive ─────────────────────────
    file_name = os.path.join(DRIVE_FOLDER, f"buildings_{hood_id}.geojson")

    if os.path.exists(file_name):
        # File already cached on Drive — load directly from there.
        print(f"  [Drive] Loading hood {hood_id} from Drive cache…")
        buildings = gpd.read_file(file_name)
    else:
        # First run: fetch from OSM, then persist to Drive for future runs.
        print(f"  [Drive] Fetching hood {hood_id} from OSM and saving to Drive…")
        pts_lonlat = [(lon, lat) for (lat, lon) in hood_border]
        poly = ShapelyPolygon(pts_lonlat)
        buildings = ox.features_from_polygon(poly, tags={"building": True})
        buildings.to_file(file_name, driver="GeoJSON")
        print(f"  [Drive] Saved → {file_name}")
    # ─────────────────────────────────────────────────────────────────────────

    buildings = buildings.to_crs(epsg=2039)
    buildings["area_m2"] = buildings.area

    buildings["point"] = buildings.geometry.representative_point()
    total_area = buildings["area_m2"].sum()
    buildings["estimated_people"] = (buildings["area_m2"] / total_area) * hood_population
    return buildings

In [ ]:
# @title function that computes neighborhood areas
def compute_neighborhood_areas(hood_borders):
    """
    Parameters:
        hood_borders: list of lists, each containing (lat, lon) tuples
    Returns:
        area_for_each_neighberhood: list of areas in m² for each neighbourhood
    """
    area_for_each_neighberhood = []

    for hood_border in hood_borders:
        pts_lonlat = [(lon, lat) for (lat, lon) in hood_border]
        poly = ShapelyPolygon(pts_lonlat)
        hood_poly = gpd.GeoSeries([poly], crs="EPSG:4326").to_crs(epsg=2039)
        area_for_each_neighberhood.append(hood_poly.area.iloc[0] / 1000000)

    return area_for_each_neighberhood

In [ ]:
def formula1(network: Network):
  return (network.K**2 + network.K**3) / ((network.L)**2 + network.network_inertia)

def formula2(network: Network):
  return (network.K**2 / (network.L**2) + ((network.K**3) / (network.network_inertia)))

def formula3(network: Network):
  beersheva_area = sum(area_for_each_neighberhood_beer_sheva)
  neighborhoods_in_beersheva = 19

  part1 = (network.K**2) * (beersheva_area) #part 1 = 5757
  part2 = (2* (neighborhoods_in_beersheva**2) * network.gravitating * network.L)
  part3 = ((network.K**3) * (network.attracting**2) * network.all_of_demand_zones_q)
  part4 = (2* (neighborhoods_in_beersheva**3) * network.network_inertia)

  return  (((part1 / part2) + (part2/part3)) * 100 )

def formula4(network: Network): #change miu(u) to only gravitating zones area.
  beersheva_area = sum(area_for_each_neighberhood_beer_sheva)
  neighborhoods_in_beersheva = 19

  part1 = (network.K**2) * (network.gravitating_demand_zones_area)
  part2 = (2* (neighborhoods_in_beersheva**2) * network.gravitating * network.L)


  part3 = ((network.K**2) * (network.attracting**2) * network.all_of_demand_zones_q)
  part4 = (2* (neighborhoods_in_beersheva**2) * network.network_inertia)

  return ( ((part1 / part2) + (part3 / part4)) * 100 )

def formula5(network: Network): #put beersheva area in denominator
  beersheva_area = sum(area_for_each_neighberhood_beer_sheva)
  neighborhoods_in_beersheva = 19
  part1 = (network.K**2) * (network.gravitating_demand_zones_area)
  part2 = (2* (neighborhoods_in_beersheva**2) * network.gravitating * network.L * beersheva_area)


  part3 = ((network.K**2) * (network.attracting**2) * network.all_of_demand_zones_q)
  part4 = (2* (neighborhoods_in_beersheva**2) * network.network_inertia)

  return ( ((part1 / part2) + (part3 / part4)) * 100 )

def formula6(network: Network): #put beersheva area in denominator
  beersheva_area = sum(area_for_each_neighberhood_beer_sheva)
  neighborhoods_in_beersheva = 19
  part1 = ((network.K**2) * network.gravitating * network.L)
  part2 = (2* (neighborhoods_in_beersheva**2) * network.gravitating_demand_zones_area)

  part3 = ((network.K**2) * (network.attracting**2) * network.all_of_demand_zones_q)
  part4 = (2* (neighborhoods_in_beersheva**2) * network.network_inertia)

  return ( ((part1 / part2) + (part3 / part4)) * 100 )

'''
K = 7
K^2 = 49
neighborhoods_in_beersheva^3 = 6859
L = 8.88402909343501
Q value of all gravitating zones = 91974.99999999996
network_inertia = 12612.903164293783
gravitating = 0.8
attracting = 0.4
'''

'\nK = 7\nK^2 = 49\nneighborhoods_in_beersheva^3 = 6859\nL = 8.88402909343501\nQ value of all gravitating zones = 91974.99999999996\nnetwork_inertia = 12612.903164293783\ngravitating = 0.8\nattracting = 0.4\n'

# Code

In [ ]:
#creating the network.
kav_3_network = Network()

for lat,lon in kav_3_nodes:
    kav_3_network.add_node((lat, lon))

In [ ]:
#creating demand points.
points = Points()

from tqdm import tqdm

for i in tqdm(range(len(hood_border))):
    df = get_demand_points_from_neighberhood(hood_border[i], hood_population[i], i)

    points_latlon = gpd.GeoSeries(df["point"], crs="EPSG:2039").to_crs(epsg=4326)


    points.add_area_in_demand_zone(i + 1, area_for_each_neighberhood_beer_sheva[i])

    for idx, point_geom in points_latlon.items():
      lon = point_geom.x
      lat = point_geom.y
      demand = df.loc[idx, "estimated_people"]

      points.add_demand_point(i + 1, lat, lon, demand)


  5%|▌         | 1/19 [00:00<00:03,  5.72it/s]

  [Drive] Loading hood 0 from Drive cache…
  [Drive] Loading hood 1 from Drive cache…
  [Drive] Loading hood 2 from Drive cache…


 16%|█▌        | 3/19 [00:00<00:01, 11.02it/s]

  [Drive] Loading hood 3 from Drive cache…
  [Drive] Loading hood 4 from Drive cache…


 26%|██▋       | 5/19 [00:00<00:01,  9.66it/s]

...

 89%|████████▉ | 17/19 [00:03<00:00,  3.57it/s]

  [Drive] Loading hood 17 from Drive cache…


 95%|█████████▍| 18/19 [00:03<00:00,  3.10it/s]

  [Drive] Loading hood 18 from Drive cache…


100%|██████████| 19/19 [00:03<00:00,  4.88it/s]


In [ ]:
kav_3_network.calculate_network(points)

# Sanity Checks

In [ ]:
print("K =", kav_3_network.K)
print("L =", kav_3_network.L)
print("Q value of all zones =", kav_3_network.all_of_demand_zones_q)
print("Q value of all gravitating zones =", kav_3_network.gravitating_demand_zones_area)
print("L^2 =", kav_3_network.L**2)
print("L^3 =", kav_3_network.L**3)
print("network_inertia =", kav_3_network.network_inertia)
print("prev gravitating q value: ", 91974.99999999996)

K = 7
L = 8.88402909343501
Q value of all zones = 236998.99999999994
Q value of all gravitating zones = 10.736824138992235
L^2 = 78.9259729329997
L^3 = 701.1806397644335
network_inertia = 12612.903164293783
prev gravitating q value:  91974.99999999996


In [ ]:
formula1(kav_3_network)

np.float64(0.030886013021575676)

In [ ]:
formula2(kav_3_network)

np.float64(0.6480292923712856)

In [ ]:
formula3(kav_3_network)

np.float64(25.297453641832956)

In [ ]:
formula4(kav_3_network)

np.float64(30.65638288862569)

In [ ]:
formula5(kav_3_network)

np.float64(20.791374968407496)

In [ ]:
formula6(kav_3_network)

np.float64(24.89621261192132)

# using ipyleaflet map

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

from ipyleaflet import Map, Marker, Polyline, Polygon, MeasureControl
from ipywidgets import Button, VBox, HBox, Output
from IPython.display import display, clear_output

# ---------------------------------------------------------
# 1. SETTINGS
# ---------------------------------------------------------
MAP_CENTER      = (31.24524, 34.78958)
MAP_ZOOM        = 13
MIN_ROUTE_KM    = 3.0
EDIT_COLOR      = "red"
SNAPSHOT_COLORS = ["green", "orange", "purple", "blue", "cadetblue", "darkred"]
# ---------------------------------------------------------
# 3. REBUILD NETWORK FROM CURRENT MARKERS
# ---------------------------------------------------------
def build_network_from_nodes(nodes, points):
    net = Network()
    for node in nodes:
        net.add_node(tuple(node))

    if len(nodes) >= 2:
        net._Network__calculate_network(points)
    return net

# ---------------------------------------------------------
# 4. EDITOR
# ---------------------------------------------------------
class NetworkEditor:
    def __init__(self, neighborhoods_data, points):
        self.neighborhoods_data = neighborhoods_data
        self.points = points

        self.edit_markers = []
        self.edit_segments = []

        self.saved_snapshots = []
        self.saved_layers = []

        self.map = Map(center=MAP_CENTER, zoom=MAP_ZOOM)
        self.map.add_control(
            MeasureControl(
                position="topleft",
                active_color="orange",
                primary_length_unit="meters"
            )
        )

        self.output = Output()

        self.save_btn = Button(description="Save Snapshot", button_style="success")
        self.new_btn = Button(description="New Editable Network", button_style="warning")
        self.undo_btn = Button(description="Undo Last Node", button_style="info")
        self.clear_saved_btn = Button(description="Clear Saved Snapshots", button_style="danger")

        self.save_btn.on_click(self.on_save)
        self.new_btn.on_click(self.on_new_network)
        self.undo_btn.on_click(self.on_undo_last_node)
        self.clear_saved_btn.on_click(self.on_clear_saved)

        self.map.on_interaction(self.on_map_click)

        self.draw_neighborhoods()
        self.refresh_output("Click the map to add the first node.")

    # ---------------------------------------------------------
    # HELPERS
    # ---------------------------------------------------------
    def get_current_nodes(self):
        return [tuple(marker.location) for marker in self.edit_markers]

    def get_current_network(self):
        nodes = self.get_current_nodes()
        if len(nodes) < 2:
            return None
        return build_network_from_nodes(nodes, self.points)

    def draw_neighborhoods(self):
        colors = ["green", "orange", "purple", "blue"]
        for i, coords in enumerate(self.neighborhoods_data):
            self.map.add_layer(
                Polygon(
                    locations=coords,
                    color=colors[i % len(colors)],
                    fill_color=colors[i % len(colors)],
                    fill_opacity=0.2,
                    weight=2
                )
            )

    def add_marker(self, lat, lon):
        marker = Marker(
            location=(lat, lon),
            draggable=True,
            title=f"Node {len(self.edit_markers) + 1}"
        )

        def _on_location_change(change):
            if change["name"] == "location":
                self.refresh_edit_line()
                self.refresh_output("Node moved.")

        marker.observe(_on_location_change, names="location")

        self.edit_markers.append(marker)
        self.map.add_layer(marker)
        self.refresh_marker_titles()
        self.refresh_edit_line()

    def refresh_marker_titles(self):
        for i, marker in enumerate(self.edit_markers, start=1):
            marker.title = f"Node {i}"

    def clear_edit_segments(self):
        for seg in self.edit_segments:
            self.map.remove_layer(seg)
        self.edit_segments = []

    def refresh_edit_line(self):
        self.clear_edit_segments()

        nodes = self.get_current_nodes()
        if len(nodes) < 2:
            return

        for i in range(len(nodes) - 1):
            seg = Polyline(
                locations=[nodes[i], nodes[i + 1]],
                color=EDIT_COLOR,
                weight=4,
                opacity=0.9
            )
            self.map.add_layer(seg)
            self.edit_segments.append(seg)

    def add_saved_snapshot_to_map(self, nodes, color):
        snapshot_segments = []

        for i in range(len(nodes) - 1):
            seg = Polyline(
                locations=[nodes[i], nodes[i + 1]],
                color=color,
                weight=3,
                opacity=0.65
            )
            self.map.add_layer(seg)
            snapshot_segments.append(seg)

        self.saved_layers.extend(snapshot_segments)

    def refresh_output(self, message=""):
        nodes = self.get_current_nodes()

        with self.output:
            clear_output(wait=True)

            print("Current editable network")
            print("-" * 45)
            print(message)

            if len(nodes) == 0:
                print("No nodes yet.")
            else:
                print(f"Nodes: {len(nodes)}")

            if len(nodes) >= 2:
                net = self.get_current_network()
                score1 = formula3(net)
                score = formula4(net)
                score2 = formula5(net)
                score3 = formula6(net)

                print(f"L = {getattr(net, 'L', 0):.2f} km")
                print(f"K = {getattr(net, 'K', 0)}")
                print(f"Network inertia = {getattr(net, 'network_inertia', 0):.4f}")
                print(f"article formula3 = {score1:.2f}%")
                print(f"new purposed formula4 = {score:.2f}%")
                print(f"new purposed formula5 = {score2:.2f}%")
                print(f"new purposed formula6 = {score3:.2f}%")

                if getattr(net, "L", 0) < MIN_ROUTE_KM:
                    remaining = MIN_ROUTE_KM - getattr(net, "L", 0)
                    print(f"Keep drawing... {remaining:.2f} km more before minimum route length")
            else:
                print("Add at least 2 nodes to compute the score.")

            if self.saved_snapshots:
                print("\nSaved snapshots")
                print("-" * 45)
                best_score = max(s["score"] for s in self.saved_snapshots)

                print(f"{'Ver':>4}  {'Score':>10}  {'L(km)':>8}  {'K':>4}  {'Nodes':>5}")
                print(f"{'-'*4}  {'-'*10}  {'-'*8}  {'-'*4}  {'-'*5}")

                for s in self.saved_snapshots:
                    best_mark = "  <-- best" if s["score"] == best_score else ""
                    print(
                        f"{s['version']:>4}  "
                        f"{s['score']:>9.2f}%  "
                        f"{s['L']:>8.2f}  "
                        f"{s['K']:>4}  "
                        f"{s['nodes']:>5}"
                        f"{best_mark}"
                    )

    # ---------------------------------------------------------
    # EVENTS
    # ---------------------------------------------------------
    def on_map_click(self, **kwargs):
        if kwargs.get("type") != "click":
            return

        lat, lon = kwargs["coordinates"]
        self.add_marker(lat, lon)
        self.refresh_output(f"Added node {len(self.edit_markers)} at ({lat:.5f}, {lon:.5f})")

    def on_save(self, _):
        nodes = self.get_current_nodes()
        cur_net = self.get_current_network()

        if len(nodes) < 2:
            self.refresh_output("Add at least 2 nodes before saving.")
            return
        elif cur_net.L < 2:
          self.refresh_output("network needs to be at least 2km to save")
          return


        net = self.get_current_network()
        score = formula3(net)

        version = len(self.saved_snapshots) + 1
        color = SNAPSHOT_COLORS[(version - 1) % len(SNAPSHOT_COLORS)]

        self.saved_snapshots.append({
            "version": version,
            "score": score,
            "nodes": len(nodes),
            "L": round(getattr(net, "L", 0), 2),
            "K": getattr(net, "K", 0),
            "node_locations": list(nodes),
            "color": color
        })

        self.add_saved_snapshot_to_map(nodes, color)
        self.refresh_output(f"Saved snapshot #{version}.")

    def on_new_network(self, _):
        self.clear_edit_segments()

        for marker in self.edit_markers:
            self.map.remove_layer(marker)

        self.edit_markers = []
        self.refresh_output("Started a new editable network. Saved snapshots stayed on the map.")

    def on_undo_last_node(self, _):
        if not self.edit_markers:
            self.refresh_output("Nothing to undo.")
            return

        last_marker = self.edit_markers.pop()
        self.map.remove_layer(last_marker)

        self.refresh_marker_titles()
        self.refresh_edit_line()
        self.refresh_output("Removed last node.")

    def on_clear_saved(self, _):
        for layer in self.saved_layers:
            self.map.remove_layer(layer)

        self.saved_layers = []
        self.saved_snapshots = []
        self.refresh_output("Cleared all saved snapshots from the map.")

    # ---------------------------------------------------------
    # DISPLAY
    # ---------------------------------------------------------
    def show(self):
        display(
            VBox([
                self.map,
                HBox([self.save_btn, self.new_btn, self.undo_btn, self.clear_saved_btn]),
                self.output
            ])
        )

# ---------------------------------------------------------
# 5. RUN
# ---------------------------------------------------------
editor = NetworkEditor(neighborhoods_data=hood_border, points=points)
editor.show()